# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR^2 dataset package using the `mlcroissant` library. We'll examine clinicopathological and molecular variables of cancer survivors with secondary colorectal cancer.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Let's load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Let's review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

**Note:** The list of record sets may be empty in some metadata. We'll attempt to enumerate them and print field information per set if available.

In [ ]:
# Display available record sets
record_sets = dataset.record_sets
print("Record Sets Found:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    # Fields overview
    fields = rs['fields'] if 'fields' in rs else []
    for f in fields:
        print(f"    - Field @id: {f['@id']} | Name: {f.get('name', 'N/A')} | DataType: {f.get('dataType', 'N/A')}")

## 3. Data Extraction
We'll load data from each record set into a DataFrame for analysis, referencing sets and fields by their `@id`.

In [ ]:
# Build list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print("Available RecordSet @ids:", record_set_ids)

# Load each record set into dataframes
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
        print("Columns:", df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
We'll apply common processing steps such as filtering, normalization, and grouping. All fields are referenced by their `@id`.

**Example:** We'll use the first available record set for demonstration, select the first numeric field where possible.

In [ ]:
# Select the first record set and its numeric field
if len(record_set_ids) == 0:
    print("No record sets available.")
else:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    # Try to select a numeric field by inspecting columns
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric fields detected in DataFrame.")
    else:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a field that is not numeric
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric group field found.")

## 5. Visualization
Let's visualize data distributions and relationships between fields. All fields and sets are referenced by `@id`.

In [ ]:
# Visualize numeric field distribution
if len(record_set_ids) > 0 and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # If a group field is available, visualize group means
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load the FAIR^2 dataset package using Croissant metadata and `mlcroissant`, inspect its structure via `@id` references, extract data for analysis, apply common EDA steps, and visualize key numeric and categorical attributes. This workflow can be adapted for deeper clinical or biomarker investigations.